In [9]:
# Opdrachten: One-Hot Encoding, Ordinal Encoding, Binning en Normalisatie

categories = ["red", "blue", "green", "blue", "red"]

# Stap 1: unieke waarden (gesorteerd voor consistente volgorde)
unique = sorted(set(categories))
print("Unieke waarden:", unique)

# Stap 2 & 3: maak one-hot vector voor elke categorie
def one_hot_list(categories, unique):
    result = []
    for item in categories:
        vector = [0] * len(unique)
        index = unique.index(item)   # .index() zoekt lineair door de lijst
        vector[index] = 1
        result.append(vector)
    return result

encoded = one_hot_list(categories, unique)

print(f"\n{'Categorie':<10} {'One-Hot Vector'}")
print("-" * 30)
for cat, vec in zip(categories, encoded):
    print(f"{cat:<10} {vec}")



Unieke waarden: ['blue', 'green', 'red']

Categorie  One-Hot Vector
------------------------------
red        [0, 0, 1]
blue       [1, 0, 0]
green      [0, 1, 0]
blue       [1, 0, 0]
red        [0, 0, 1]


In [10]:
# ## Opdracht 2: Datastructuren vergelijken bij grote dataset

import random
import time

random.seed(42)
colors = ["red", "blue", "green", "yellow", "purple"]
categories = [random.choice(colors) for _ in range(20000)]

unique_sorted = sorted(set(categories))

# Methode 1: LIST met .index()
def one_hot_with_list(categories, unique):
    result = []
    for item in categories:
        vector = [0] * len(unique)
        vector[unique.index(item)] = 1   # O(n) zoektijd per aanroep
        result.append(vector)
    return result

start = time.time()
result_list = one_hot_with_list(categories, unique_sorted)
time_list = time.time() - start
print(f"Methode 1 – List .index()  : {time_list:.4f} sec")

# Methode 2: DICTIONARY (hash map)
def one_hot_with_dict(categories, unique):
    # Bouw één keer een mapping: waarde → index  →  O(1) lookup
    index_map = {val: idx for idx, val in enumerate(unique)}
    result = []
    for item in categories:
        vector = [0] * len(unique)
        vector[index_map[item]] = 1      # O(1) opzoeken
        result.append(vector)
    return result

start = time.time()
result_dict = one_hot_with_dict(categories, unique_sorted)
time_dict = time.time() - start
print(f"Methode 2 – Dictionary     : {time_dict:.4f} sec")

Methode 1 – List .index()  : 0.0065 sec
Methode 2 – Dictionary     : 0.0049 sec


In [11]:

# Methode 3: SET (alleen voor unieke waarden, NIET voor encoden)
# Een set heeft geen volgorde → we kunnen er geen index uit halen.
# We gebruiken de set alleen om unieke labels te vinden, daarna alsnog een dict.
def one_hot_with_set(categories):
    unique_set = sorted(set(categories))          # set voor unieke waarden
    index_map  = {val: idx for idx, val in enumerate(unique_set)}
    result = []
    for item in categories:
        vector = [0] * len(unique_set)
        vector[index_map[item]] = 1
        result.append(vector)
    return result

start = time.time()
result_set = one_hot_with_set(categories)
time_set = time.time() - start
print(f"Methode 3 – Set + dict     : {time_set:.4f} sec")

# Samenvatting
print("\n── Tijdsvergelijking ──────────────────────────────────")
print(f"  List   : {time_list:.4f} sec  (langzaamst – O(n) per lookup)")
print(f"  Dict   : {time_dict:.4f} sec  (snel      – O(1) per lookup)")
print(f"  Set+dict:{time_set:.4f} sec  (snel      – set enkel voor unieke waarden)")
speedup = time_list / time_dict
print(f"\n  Dictionary is ~{speedup:.1f}x sneller dan list met .index()")

Methode 3 – Set + dict     : 0.0049 sec

── Tijdsvergelijking ──────────────────────────────────
  List   : 0.0065 sec  (langzaamst – O(n) per lookup)
  Dict   : 0.0049 sec  (snel      – O(1) per lookup)
  Set+dict:0.0049 sec  (snel      – set enkel voor unieke waarden)

  Dictionary is ~1.3x sneller dan list met .index()


In [12]:

# %% [markdown]
# ### Analyse & vragen
#
# | Vraag | Antwoord |
# |---|---|
# | **Snelste datastructuur?** | Dictionary (O(1) hash-lookup) |
# | **Dictionary vs list?** | Dict gebruikt hashing → directe toegang op sleutel, list zoekt lineair |
# | **Waarom is .index() traag?** | Doorzoekt de lijst element voor element (O(n)); bij 20 000 items × 5 labels = 100 000 vergelijkingen |
# | **Waarom helpt set niet bij encoden?** | Set heeft **geen volgorde** en geen integer-index, dus je kunt geen positie opzoeken |
# | **Miljoenen elementen?** | List schaalt kwadratisch, dict blijft lineair → verschil wordt enorm |
# | **Meer unieke labels?** | .index() wordt extra traag; dict blijft O(1) ongeacht het aantal labels |
# | **Real-world keuze?** | Dictionary (of NumPy/pandas equivalent) – snelste en meest geheugenefficiënt |

# ── Bonus: schaal naar 1 miljoen ──────────────────────────────────────────────
# %%
large = [random.choice(colors) for _ in range(1_000_000)]

start = time.time()
_ = one_hot_with_dict(large, unique_sorted)
print(f"1 000 000 items – dict  : {time.time() - start:.4f} sec")

# List met .index() op 1M items duurt erg lang; uncomment om te proberen:
# start = time.time()
# _ = one_hot_with_list(large, unique_sorted)
# print(f"1 000 000 items – list  : {time.time() - start:.4f} sec")

1 000 000 items – dict  : 0.3667 sec


In [13]:
# ## Opdracht 3: Ordinal Encoding

# %%
reviews = ["goed", "matig", "matig", "slecht", "uitstekend"]

# Volgorde bepalen op basis van sentiment (laag → hoog)
ordinal_map = {
    "slecht":     1,
    "matig":      2,
    "goed":       3,
    "uitstekend": 4,
}

encoded_ordinal = [ordinal_map[r] for r in reviews]

print("Ordinal Encoding")
print("-" * 30)
for review, code in zip(reviews, encoded_ordinal):
    print(f"  {review:<12} → {code}")

print("\nOrdinal map:", ordinal_map)


Ordinal Encoding
------------------------------
  goed         → 3
  matig        → 2
  matig        → 2
  slecht       → 1
  uitstekend   → 4

Ordinal map: {'slecht': 1, 'matig': 2, 'goed': 3, 'uitstekend': 4}


In [14]:
# ## Opdracht 4: Binning (equal-frequency)

# %%
import math

leeftijden = [12, 18, 22, 25, 30, 35, 40, 45, 50, 55, 60, 70]

def equal_frequency_binning(data, n_bins=3, labels=None):
    """
    Deelt data in n gelijke groepen op basis van frequentie (aantal elementen).
    Geeft een lijst van (waarde, label) terug.
    """
    if labels is None:
        labels = list(range(1, n_bins + 1))

    sorted_data = sorted(data)
    n = len(sorted_data)
    bin_size = math.ceil(n / n_bins)

    # Maak grenzen: elke bin bevat ~bin_size elementen
    boundaries = []
    for i in range(n_bins):
        chunk = sorted_data[i * bin_size : (i + 1) * bin_size]
        if chunk:
            boundaries.append(max(chunk))

    def get_bin(value):
        for i, boundary in enumerate(boundaries):
            if value <= boundary:
                return labels[i]
        return labels[-1]

    return [(val, get_bin(val)) for val in data]

bin_labels = ["Jong", "Midden", "Oud"]
result = equal_frequency_binning(leeftijden, n_bins=3, labels=bin_labels)

print("Equal-Frequency Binning (3 categorieën)")
print("-" * 35)
for leeftijd, label in result:
    print(f"  Leeftijd {leeftijd:>3}  →  {label}")

Equal-Frequency Binning (3 categorieën)
-----------------------------------
  Leeftijd  12  →  Jong
  Leeftijd  18  →  Jong
  Leeftijd  22  →  Jong
  Leeftijd  25  →  Jong
  Leeftijd  30  →  Midden
  Leeftijd  35  →  Midden
  Leeftijd  40  →  Midden
  Leeftijd  45  →  Midden
  Leeftijd  50  →  Oud
  Leeftijd  55  →  Oud
  Leeftijd  60  →  Oud
  Leeftijd  70  →  Oud


In [15]:
# ## Opdracht 5: Normalisatie (min-max)

# %%
waarden = [18, 22, 30, 45, 60]

def min_max_normalize(data):
    """Schaalt waarden lineair naar het bereik [0, 1]."""
    min_val = min(data)
    max_val = max(data)
    return [(x - min_val) / (max_val - min_val) for x in data]

genormaliseerd = min_max_normalize(waarden)

print("Min-Max Normalisatie")
print(f"  min = {min(waarden)}, max = {max(waarden)}")
print("-" * 35)
for orig, norm in zip(waarden, genormaliseerd):
    print(f"  {orig:>5}  →  {norm:.4f}")

# Formule ter referentie
print("\nFormule:  x_norm = (x - min) / (max - min)")

Min-Max Normalisatie
  min = 18, max = 60
-----------------------------------
     18  →  0.0000
     22  →  0.0952
     30  →  0.2857
     45  →  0.6429
     60  →  1.0000

Formule:  x_norm = (x - min) / (max - min)
